In [1]:
# ============================================================
# CS781 Capstone - PHASE 6: Abstractive Legal Summarization (primary metric BERTScore F1)
# Decoder-only seq2seq with YOUR Phase-1 LM:   [selected document tokens] [SEP] [summary] [EOS]
# Loss only on the summary tokens. Long documents: choose which part of the judgment the model reads
# (head / head+tail / TF-IDF "importance" sentence selection). Everything from scratch in plain PyTorch.
#
# One notebook for all experiments - change Cell 1 only:
#   FT_METHOD: "lora" | "qlora" | "lorma" | "adalora" | "houlsby" | "pfeiffer" | "prefix" | "prompt" | "full" | "partial"
#   Knowledge distillation: set KD_TEACHER to a trained Phase-6 run's best_summ.pt
#   Decoding-only experiment: set RESUME_CKPT to a best_summ.pt -> no training, just generate again
# ============================================================

# %% 0. IMPORTS
import os, re, json, math, time, glob, random, zipfile
from collections import Counter
from types import SimpleNamespace
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import PreTrainedTokenizerFast   # loads YOUR tokenizer file only

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = None

# %% 1. CONFIG  <-------------------------------------------------------- EDIT HERE
QUICK_TEST = False          # True = few docs, 1 epoch, short generation - to check the pipeline

MODEL_TYPE = "llama"        # "gpt" (Model 1) | "llama" (Model 2) | "gqa" (Model 3) | "hybrid" (Model 4)
PHASE1_CKPT = "/kaggle/input/datasets/kavyajain123987/phase1-models/best mdl2 phase1.pt"
TOKENIZER = "/kaggle/input/datasets/kavyajain123987/phase1/tokenizer (1).json"

FT_METHOD = "qlora"          # lora | qlora | lorma | adalora | houlsby | pfeiffer | prefix | prompt | full | partial
FREEZE_BOTTOM = 4           # partial: freeze embeddings + this many lowest blocks
LORA_R, LORA_ALPHA = 16, 32 # lora / qlora / lorma  (generation needs more capacity than classification)
ADA_R_INIT, ADA_R_TARGET = 24, 8
ADAPTER_DIM = 64
N_PROMPT = 32               # prefix / prompt virtual tokens (always reserved in the context, so KD works)
QUANT_BLOCK = 64            # qlora 4-bit block size

# ---- long-document input ----
SECTION = "select"          # "select" (TF-IDF important sentences, in original order) | "head" | "headtail"
HEAD_FRAC = 0.6             # headtail: share of the source budget taken from the start
TGT_LEN = "auto"            # summary tokens used in training ("auto" = 75th percentile of train summaries, 128..448)
SEP_TEXT = "\n\nSUMMARY:\n"

# ---- training ----
MAX_TRAIN_DOCS = 8000
EPOCHS = 3
LR_BODY = 5e-5              # full / partial
LR_NEW = 1e-3               # LoRA / adapters / prompts ...  (prefix/prompt: try 3e-3)
WEIGHT_DECAY = 0.01
MICRO_BATCH = 4
GRAD_ACCUM = 4              # effective batch = 16 documents
DEV_LOSS_DOCS = 200         # dev loss (teacher forcing) used for model selection
SEED = 42

# ---- decoding ----
MAX_NEW = "auto"            # generated tokens ("auto" = TGT_LEN)
MIN_NEW = "auto"            # no EOS before this many tokens ("auto" = 25th percentile of summaries)
DECODING = "greedy"         # "greedy" | "sample" (top-p)
TOP_P, TEMPERATURE = 0.9, 0.7
REPETITION_PENALTY = 1.0    # >1 penalises tokens already GENERATED (1.1-1.2 if summaries loop; can hurt "the", "of")
NO_REPEAT_NGRAM = 4         # never repeat a 4-gram inside the summary (0 = off)
GEN_BATCH = 16
GEN_DEV_DOCS = 60           # dev docs summarised at the end for ROUGE / length / redundancy analysis

# ---- knowledge distillation (mandatory experiment) ----
KD_TEACHER = None           # path to a trained Phase-6 run's best_summ.pt (file, .zip, or extracted folder)
KD_KEEP_EVERY = 2           # student = Phase-1 model with every 2nd block (half depth), fully fine-tuned
KD_T, KD_ALPHA = 2.0, 0.5   # loss = ALPHA * CE(gold summary) + (1-ALPHA) * T^2 * KL(teacher || student) per token

RESUME_CKPT = None          # a best_summ.pt: skip training, only decode (decoding-strategy experiments)

OUTPUT_DIR = f"/kaggle/working/phase6_{MODEL_TYPE}_{FT_METHOD}" + ("_kd" if KD_TEACHER else "") + ("_decode" if RESUME_CKPT else "")
os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

# %% 2. DOWNLOAD PHASE-6 DATA + LOOK AT IT
from huggingface_hub import snapshot_download
DATA_DIR = snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset", token=HF_TOKEN,
                             allow_patterns=["Phase_6*/*", "Phase_6*/**"], local_dir="/tmp/cs781_data")
P6 = next((p for p in sorted(glob.glob(os.path.join(DATA_DIR, "Phase_6*"))) if os.path.isdir(p)), None)
assert P6, f"no Phase_6 folder in {DATA_DIR}"
FILES = sorted(glob.glob(os.path.join(P6, "**", "*.json"), recursive=True))
for f in FILES:
    print(f, f"{os.path.getsize(f) / 1e6:.1f} MB")

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def peek(obj, n=300):
    s = json.dumps(obj, ensure_ascii=False)
    return s[:n] + ("..." if len(s) > n else "")

SENT_SPLIT = re.compile(r"(?<=[.!?;:])\s+(?=[A-Z0-9(\"'\[])")

def parse(obj):
    """-> list of {"id", "sents" (list of sentences), "summary" (str or None)}"""
    if isinstance(obj, dict) and "data" in obj and isinstance(obj["data"], (list, dict)):
        obj = obj["data"]
    items = obj.items() if isinstance(obj, dict) else enumerate(obj)
    out = []
    for k, d in items:
        if isinstance(d, str):
            d = {"document": d}
        if not isinstance(d, dict):
            continue
        doc = next((d[x] for x in ("document", "text", "judgment", "source", "doc", "content") if x in d), None)
        summ = next((d[x] for x in ("summary", "abstract", "target", "reference", "headnote") if x in d), None)
        if isinstance(doc, list):
            sents = [str(s).strip() for s in doc if str(s).strip()]
        else:
            sents = [s.strip() for s in SENT_SPLIT.split(str(doc or "")) if s.strip()]
        if isinstance(summ, list):
            summ = " ".join(map(str, summ))
        out.append({"id": str(d.get("id", d.get("doc_id", d.get("case_id", k)))), "sents": sents or [""],
                    "summary": re.sub(r"\s+", " ", summ).strip() if isinstance(summ, str) else None})
    return out

splits = {"train": [], "dev": [], "test": []}
for path in FILES:
    raw = load_json(path)
    name = os.path.basename(path).lower()
    split = "test" if "test" in name else "dev" if ("dev" in name or "val" in name) else "train"
    first = raw[0] if isinstance(raw, list) else next(iter(raw.items()))
    print(f"\n{os.path.relpath(path, P6)} -> {split}: {type(raw).__name__} with {len(raw)} entries | first: {peek(first)}")
    splits[split] += parse(raw)
assert splits["test"] and any(len(" ".join(d["sents"])) for d in splits["test"]), "test docs not parsed - check the format above"
splits["train"] = [d for d in splits["train"] if d["summary"]]
splits["dev"] = [d for d in splits["dev"] if d["summary"]]
random.Random(SEED).shuffle(splits["train"])
if not splits["dev"]:                       # no dev file released -> hold out part of train
    n_dev = min(300, max(20, len(splits["train"]) // 20))
    splits["dev"], splits["train"] = splits["train"][:n_dev], splits["train"][n_dev:]
    print(f"\nno dev file - held out {n_dev} training documents as dev")
for name, ds in splits.items():
    w = [len(" ".join(d["sents"]).split()) for d in ds]
    sw = [len(d["summary"].split()) for d in ds if d["summary"]]
    print(f"{name}: {len(ds)} docs | document words median {int(np.median(w))}, max {max(w)}"
          + (f" | summary words median {int(np.median(sw))}, p90 {int(np.percentile(sw, 90))}" if sw else ""))
if MAX_TRAIN_DOCS:
    splits["train"] = splits["train"][:MAX_TRAIN_DOCS]
if QUICK_TEST:
    splits["train"], splits["dev"] = splits["train"][:40], splits["dev"][:12]
    EPOCHS, GEN_DEV_DOCS = 1, 6
print(f"using {len(splits['train'])} training docs, {len(splits['dev'])} dev docs, {len(splits['test'])} test docs")

# %% 3. TOKENIZE: summary + the part of the document the model reads
tokenizer = PreTrainedTokenizerFast(tokenizer_file=TOKENIZER)
tokenizer.model_max_length = int(1e9)
EOS = 0                                       # <|endoftext|> in your legal BPE
SEP = tokenizer.encode(SEP_TEXT, add_special_tokens=False)
def as_ckpt_file(path):
    """Kaggle sometimes auto-extracts an uploaded .pt (it is a zip inside) into a FOLDER.
    If `path` is such a folder, re-pack it into a normal .pt file that torch.load can read."""
    if not os.path.isdir(path):
        return path
    root = next((r for r, _, f in os.walk(path) if "data.pkl" in f), None)
    assert root, f"{path} is a folder without data.pkl - not an extracted checkpoint"
    out = os.path.join("/kaggle/working", os.path.basename(path.rstrip("/")) + "_repacked.pt")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:
        for r, _, files in os.walk(root):
            for f in files:
                full = os.path.join(r, f)
                z.write(full, "archive/" + os.path.relpath(full, root))
    print(f"Re-packed extracted checkpoint folder {path} -> {out}")
    return out
ck = torch.load(as_ckpt_file(PHASE1_CKPT), map_location="cpu")
P1_CFG = ck["config"]
print("Phase-1 config:", P1_CFG)
assert P1_CFG["vocab_size"] == len(tokenizer), "Tokenizer doesn't match this Phase-1 model"
CTX = P1_CFG.get("block_size", P1_CFG.get("max_seq_len"))

def enc(texts):
    return tokenizer(texts, add_special_tokens=False)["input_ids"]

t0 = time.time()
for ds in (splits["train"], splits["dev"]):
    ids = enc([d["summary"] for d in ds])
    for d, x in zip(ds, ids):
        d["tgt"] = x
tl = [len(d["tgt"]) for d in splits["train"]]
print(f"summary tokens: median {int(np.median(tl))}, p25 {int(np.percentile(tl, 25))}, "
      f"p75 {int(np.percentile(tl, 75))}, p90 {int(np.percentile(tl, 90))}")
if TGT_LEN == "auto":
    TGT_LEN = int(np.clip(32 * round(np.percentile(tl, 75) / 32), 128, 448))
if MAX_NEW == "auto":
    MAX_NEW = TGT_LEN
if MIN_NEW == "auto":
    MIN_NEW = int(np.clip(np.percentile(tl, 25), 16, MAX_NEW // 2))
if QUICK_TEST:
    MAX_NEW, MIN_NEW = min(MAX_NEW, 48), min(MIN_NEW, 16)
print(f"TGT_LEN {TGT_LEN} | MAX_NEW {MAX_NEW} | MIN_NEW {MIN_NEW} | "
      f"{100 * np.mean([x > TGT_LEN for x in tl]):.0f}% of train summaries are cut at TGT_LEN")

WORD = re.compile(r"[a-z0-9]+")

def select_sentences(sents, budget):
    """importance-based selection: TF-IDF cosine of each sentence to the whole document (+ small bonus for
    the opening and the closing sentences, where parties / final order are), keep the best sentences that
    fit the token budget, in their ORIGINAL order."""
    ids = enc([s if i == 0 else " " + s for i, s in enumerate(sents)])
    lens = [len(x) for x in ids]
    if sum(lens) <= budget:
        return [t for x in ids for t in x]
    toks = [WORD.findall(s.lower()) for s in sents]
    df = Counter(w for t in toks for w in set(t))
    N = len(sents)
    idf = {w: math.log(N / c) + 1.0 for w, c in df.items()}
    vecs = [{w: c * idf[w] for w, c in Counter(t).items()} for t in toks]
    cen = Counter()
    for v in vecs:
        cen.update(v)
    cn = math.sqrt(sum(x * x for x in cen.values())) or 1.0
    score = []
    for i, v in enumerate(vecs):
        vn = math.sqrt(sum(x * x for x in v.values())) or 1.0
        s = sum(x * cen[w] for w, x in v.items()) / (vn * cn)
        if i < 3 or i >= N - 3:
            s += 0.2
        if lens[i] < 4:
            s -= 1.0
        score.append(s)
    keep, used = set(), 0
    for i in sorted(range(N), key=lambda i: -score[i]):
        if used + lens[i] <= budget:
            keep.add(i); used += lens[i]
    return [t for i in sorted(keep) for t in ids[i]]

def source_ids(sents, budget):
    if SECTION == "select":
        return select_sentences(sents, budget)
    text = " ".join(sents)
    if SECTION == "head":
        return enc([text[: budget * 8]])[0][:budget]
    if SECTION == "headtail":
        ids = enc([text])[0]
        if len(ids) <= budget:
            return ids
        h = int(budget * HEAD_FRAC)
        return ids[:h] + ids[-(budget - h):]
    raise ValueError(SECTION)

SRC_LEN = CTX - N_PROMPT - len(SEP) - TGT_LEN - 1
assert SRC_LEN >= 128, "TGT_LEN / N_PROMPT too large for the Phase-1 context"
print(f"context {CTX} = {N_PROMPT} (reserved for prompts) + source {SRC_LEN} + SEP {len(SEP)} + summary {TGT_LEN} + EOS")
for name, ds in splits.items():
    for d in ds:
        d["src"] = source_ids(d["sents"], SRC_LEN) or [EOS]
        d["n_doc_tok_est"] = sum(len(s) for s in d["sents"]) // 4
        d["sents"] = d["sents"] if name != "train" else None       # keep text of dev/test for fallbacks
    print(f"{name}: source built in {time.time() - t0:.0f}s | "
          f"{100 * np.mean([d['n_doc_tok_est'] > SRC_LEN for d in ds]):.0f}% of documents longer than the source budget")

# %% 4. PHASE-1 MODEL DEFINITIONS (unchanged from Phase 1)
def build_model1(cfg):
    # ---------- Model 1: GPT-style ----------
    class CausalSelfAttention(nn.Module):
        def __init__(self, d_model, n_head, dropout):
            super().__init__()
            assert d_model % n_head == 0
            self.d_model = d_model
            self.n_head = n_head
            self.head_dim = d_model // n_head
            self.c_attn = nn.Linear(d_model, 3 * d_model)
            self.c_proj = nn.Linear(d_model, d_model)
            self.attn_dropout = dropout
            self.resid_dropout = nn.Dropout(dropout)

        def forward(self, x):
            B, T, C = x.shape
            qkv = self.c_attn(x)
            q, k, v = qkv.split(self.d_model, dim=2)
            q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            y = F.scaled_dot_product_attention(
                q, k, v, attn_mask=None,
                dropout_p=(self.attn_dropout if self.training else 0.0),
                is_causal=True)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            y = self.c_proj(y)
            y = self.resid_dropout(y)
            return y


    class MLP(nn.Module):
        def __init__(self, d_model, dropout):
            super().__init__()
            self.fc = nn.Linear(d_model, 4 * d_model)
            self.proj = nn.Linear(4 * d_model, d_model)
            self.dropout = nn.Dropout(dropout)

        def forward(self, x):
            x = self.fc(x)
            x = F.gelu(x)
            x = self.proj(x)
            x = self.dropout(x)
            return x


    class TransformerBlock(nn.Module):
        def __init__(self, d_model, n_head, dropout):
            super().__init__()
            self.ln_1 = nn.LayerNorm(d_model)
            self.attn = CausalSelfAttention(d_model, n_head, dropout)
            self.ln_2 = nn.LayerNorm(d_model)
            self.mlp = MLP(d_model, dropout)

        def forward(self, x):
            x = x + self.attn(self.ln_1(x))
            x = x + self.mlp(self.ln_2(x))
            return x


    class LegalGPT(nn.Module):
        def __init__(self, vocab_size, block_size, n_layer, n_head, d_model, dropout):
            super().__init__()
            self.block_size = block_size
            self.wte = nn.Embedding(vocab_size, d_model)
            self.wpe = nn.Embedding(block_size, d_model)
            self.blocks = nn.ModuleList([TransformerBlock(d_model, n_head, dropout) for _ in range(n_layer)])
            self.ln_f = nn.LayerNorm(d_model)
            self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
            self.lm_head.weight = self.wte.weight
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, idx, targets=None):
            B, T = idx.shape
            assert T <= self.block_size
            positions = torch.arange(0, T, device=idx.device)
            x = self.wte(idx) + self.wpe(positions)
            for block in self.blocks:
                x = block(x)
            x = self.ln_f(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss
    return LegalGPT(**cfg)


def build_model2(cfg):
    # ---------- Model 2: LLaMA-style ----------
    class RMSNorm(nn.Module):
        """x / sqrt(mean(x^2) + eps) * weight  (no mean subtraction, unlike LayerNorm)"""
        def __init__(self, dim, eps=1e-6):
            super().__init__()
            self.weight = nn.Parameter(torch.ones(dim))
            self.eps = eps

        def forward(self, x):
            rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
            return (x * rms) * self.weight


    class RotaryEmbedding(nn.Module):
        """RoPE: rotate q and k by position-dependent angles (relative position info)."""
        def __init__(self, dim, max_seq_len, theta=10000.0):
            super().__init__()
            self.dim = dim
            self.max_seq_len = max_seq_len
            inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
            self.register_buffer("inv_freq", inv_freq, persistent=False)
            self._build_cache(max_seq_len)

        def _build_cache(self, seq_len):
            positions = torch.arange(seq_len, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
            freqs = torch.outer(positions, self.inv_freq)
            emb = torch.cat([freqs, freqs], dim=-1)
            self.register_buffer("cos_cached", emb.cos()[None, :, :], persistent=False)
            self.register_buffer("sin_cached", emb.sin()[None, :, :], persistent=False)

        def forward(self, q, k):
            seq_len = q.size(-2)
            if seq_len > self.cos_cached.size(1):
                self._build_cache(seq_len)
            cos = self.cos_cached[:, :seq_len, :].to(q.dtype)
            sin = self.sin_cached[:, :seq_len, :].to(q.dtype)
            return apply_rotary(q, cos, sin), apply_rotary(k, cos, sin)


    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2:]
        return torch.cat((-x2, x1), dim=-1)


    def apply_rotary(x, cos, sin):
        return x * cos + rotate_half(x) * sin


    class SwiGLU(nn.Module):
        """down( SiLU(gate(x)) * up(x) ), no bias"""
        def __init__(self, d_model, ffn_dim):
            super().__init__()
            self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)

        def forward(self, x):
            return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


    class CausalSelfAttention(nn.Module):
        def __init__(self, d_model, n_heads, max_seq_len):
            super().__init__()
            assert d_model % n_heads == 0
            self.d_model = d_model
            self.n_heads = n_heads
            self.head_dim = d_model // n_heads
            self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)
            self.rope = RotaryEmbedding(self.head_dim, max_seq_len)

        def forward(self, x):
            B, T, C = x.shape
            q, k, v = self.qkv(x).chunk(3, dim=-1)
            q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            q, k = self.rope(q, k)
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=True)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            return self.out_proj(y)


    class LLaMABlock(nn.Module):
        def __init__(self, d_model, n_heads, ffn_dim, max_seq_len):
            super().__init__()
            self.attn_norm = RMSNorm(d_model)
            self.attention = CausalSelfAttention(d_model, n_heads, max_seq_len)
            self.ffn_norm = RMSNorm(d_model)
            self.ffn = SwiGLU(d_model, ffn_dim)

        def forward(self, x):
            x = x + self.attention(self.attn_norm(x))   # pre-norm
            x = x + self.ffn(self.ffn_norm(x))
            return x


    class LLaMALanguageModel(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            self.cfg = cfg
            self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
            self.blocks = nn.ModuleList([LLaMABlock(cfg.d_model, cfg.n_heads, cfg.ffn_dim, cfg.max_seq_len)
                                         for _ in range(cfg.n_layers)])
            self.final_norm = RMSNorm(cfg.d_model)
            self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
            self.lm_head.weight = self.token_embedding.weight      # weight tying
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, idx, targets=None):
            x = self.token_embedding(idx)          # no position embedding: RoPE handles position
            for block in self.blocks:
                x = block(x)
            x = self.final_norm(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            return logits, loss
        # (your generate() method is not needed: the pipeline's batched generate_batch() is used)
    return LLaMALanguageModel(SimpleNamespace(**cfg))


def build_model3(cfg):
    # ---------- Model 3: LLaMA + GQA + sliding window ----------
    class RMSNorm(nn.Module):
        """x / sqrt(mean(x^2) + eps) * weight  (no mean subtraction, unlike LayerNorm)"""
        def __init__(self, dim, eps=1e-6):
            super().__init__()
            self.weight = nn.Parameter(torch.ones(dim))
            self.eps = eps

        def forward(self, x):
            rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
            return (x * rms) * self.weight


    class RotaryEmbedding(nn.Module):
        """RoPE: rotate q and k by position-dependent angles (relative position info)."""
        def __init__(self, dim, max_seq_len, theta=10000.0):
            super().__init__()
            self.dim = dim
            self.max_seq_len = max_seq_len
            inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
            self.register_buffer("inv_freq", inv_freq, persistent=False)
            self._build_cache(max_seq_len)

        def _build_cache(self, seq_len):
            positions = torch.arange(seq_len, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
            freqs = torch.outer(positions, self.inv_freq)
            emb = torch.cat([freqs, freqs], dim=-1)
            self.register_buffer("cos_cached", emb.cos()[None, :, :], persistent=False)
            self.register_buffer("sin_cached", emb.sin()[None, :, :], persistent=False)

        def forward(self, q, k):
            seq_len = q.size(-2)
            if seq_len > self.cos_cached.size(1):
                self._build_cache(seq_len)
            cos = self.cos_cached[:, :seq_len, :].to(q.dtype)
            sin = self.sin_cached[:, :seq_len, :].to(q.dtype)
            return apply_rotary(q, cos, sin), apply_rotary(k, cos, sin)


    def rotate_half(x):
        half = x.shape[-1] // 2
        x1, x2 = x[..., :half], x[..., half:]
        return torch.cat([-x2, x1], dim=-1)


    def apply_rotary(x, cos, sin):
        return x * cos + rotate_half(x) * sin


    class SwiGLU(nn.Module):
        """down( SiLU(gate(x)) * up(x) ), no bias"""
        def __init__(self, d_model, ffn_dim):
            super().__init__()
            self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)

        def forward(self, x):
            return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


    class GQASlidingWindowAttention(nn.Module):
        """
        GQA: n_heads query heads share n_kv_heads key/value heads (8 Q -> 2 KV: each KV head serves 4 Q heads).
        Sliding window: token i attends only to tokens j with 0 <= i - j < window_size (causal + local).
        """
        def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, window_size):
            super().__init__()
            assert d_model % n_heads == 0
            assert n_heads % n_kv_heads == 0
            self.d_model = d_model
            self.n_heads = n_heads
            self.n_kv_heads = n_kv_heads
            self.head_dim = d_model // n_heads
            self.window_size = window_size
            self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
            self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
            self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)
            self.rope = RotaryEmbedding(self.head_dim, max_seq_len)
            # mask[q, k] = True -> q may attend to k   (k <= q  AND  q - k < window)
            positions = torch.arange(max_seq_len)
            distance = positions[:, None] - positions[None, :]
            self.register_buffer("window_mask", (distance >= 0) & (distance < window_size), persistent=False)

        def forward(self, x):
            B, T, C = x.shape
            q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)     # [B, 8, T, D]
            k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)  # [B, 2, T, D]
            v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)  # [B, 2, T, D]
            q, k = self.rope(q, k)
            repeat_factor = self.n_heads // self.n_kv_heads
            k = k.repeat_interleave(repeat_factor, dim=1)                                  # [B, 8, T, D]
            v = v.repeat_interleave(repeat_factor, dim=1)
            mask = self.window_mask[:T, :T]
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            return self.out_proj(y)


    class LLaMABlock(nn.Module):
        def __init__(self, d_model, n_heads, n_kv_heads, ffn_dim, max_seq_len, window_size):
            super().__init__()
            self.attn_norm = RMSNorm(d_model)
            self.attention = GQASlidingWindowAttention(d_model=d_model, n_heads=n_heads, n_kv_heads=n_kv_heads,
                                                       max_seq_len=max_seq_len, window_size=window_size)
            self.ffn_norm = RMSNorm(d_model)
            self.ffn = SwiGLU(d_model, ffn_dim)

        def forward(self, x):
            x = x + self.attention(self.attn_norm(x))   # pre-norm
            x = x + self.ffn(self.ffn_norm(x))
            return x


    class LLaMALanguageModel(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            self.cfg = cfg
            self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
            self.blocks = nn.ModuleList([
                LLaMABlock(d_model=cfg.d_model, n_heads=cfg.n_heads, n_kv_heads=cfg.n_kv_heads, ffn_dim=cfg.ffn_dim,
                           max_seq_len=cfg.max_seq_len, window_size=cfg.window_size)
                for _ in range(cfg.n_layers)])
            self.final_norm = RMSNorm(cfg.d_model)
            self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
            self.lm_head.weight = self.token_embedding.weight      # weight tying
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, idx, targets=None):
            x = self.token_embedding(idx)          # no position embedding: RoPE handles position
            for block in self.blocks:
                x = block(x)
            x = self.final_norm(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            return logits, loss
    return LLaMALanguageModel(SimpleNamespace(**cfg))


def build_model4(cfg):
    # ---------- Model 4: hybrid attention + gated conv ----------
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2:]
        return torch.cat((-x2, x1), dim=-1)


    class RotaryEmbedding(nn.Module):
        """RoPE: rotate q and k by position-dependent angles."""
        def __init__(self, dim, max_seq_len=2048, base=10000):
            super().__init__()
            self.dim = dim
            self.max_seq_len = max_seq_len
            self.base = base
            inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
            self.register_buffer("inv_freq", inv_freq, persistent=False)
            self._build_cache(max_seq_len)

        def _build_cache(self, seq_len):
            positions = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
            freqs = torch.outer(positions, self.inv_freq)
            emb = torch.cat([freqs, freqs], dim=-1)
            self.register_buffer("cos_cached", emb.cos()[None, None, :, :], persistent=False)
            self.register_buffer("sin_cached", emb.sin()[None, None, :, :], persistent=False)

        def forward(self, q, k):
            seq_len = q.shape[-2]
            if seq_len > self.cos_cached.shape[-2]:
                self._build_cache(seq_len)
            # FIX: cast to q's dtype. Otherwise fp16 q * fp32 cos -> fp32 q/k while v stays fp16,
            # and scaled_dot_product_attention can fail with a dtype mismatch.
            cos = self.cos_cached[:, :, :seq_len, :].to(q.dtype)
            sin = self.sin_cached[:, :, :seq_len, :].to(q.dtype)
            q = q * cos + rotate_half(q) * sin
            k = k * cos + rotate_half(k) * sin
            return q, k


    class RMSNorm(nn.Module):
        def __init__(self, dim, eps=1e-6):
            super().__init__()
            self.weight = nn.Parameter(torch.ones(dim))
            self.eps = eps

        def forward(self, x):
            rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
            return x * rms * self.weight


    class SwiGLU(nn.Module):
        def __init__(self, d_model, hidden_dim):
            super().__init__()
            self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
            self.w2 = nn.Linear(hidden_dim, d_model, bias=False)
            self.w3 = nn.Linear(d_model, hidden_dim, bias=False)

        def forward(self, x):
            return self.w2(F.silu(self.w1(x)) * self.w3(x))


    class CausalSelfAttention(nn.Module):
        """GLOBAL mixer: token i attends to tokens 0..i."""
        def __init__(self, d_model, n_heads, max_seq_len):
            super().__init__()
            assert d_model % n_heads == 0
            self.d_model = d_model
            self.n_heads = n_heads
            self.head_dim = d_model // n_heads
            self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)
            self.rope = RotaryEmbedding(self.head_dim, max_seq_len=max_seq_len)

        def forward(self, x):
            B, T, C = x.shape
            q, k, v = self.qkv(x).chunk(3, dim=-1)
            q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            q, k = self.rope(q, k)
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=0.0, is_causal=True)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            return self.out_proj(y)


    class GatedCausalConv(nn.Module):
        """
        LOCAL mixer: Linear(D->2D) -> (value, gate);
        value -> causal depthwise Conv1d (left padding only: position t sees t-k+1 .. t, never t+1);
        out = Linear( conv(value) * sigmoid(gate) ).
        """
        def __init__(self, d_model, kernel_size=5):
            super().__init__()
            self.d_model = d_model
            self.kernel_size = kernel_size
            self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
            self.conv = nn.Conv1d(d_model, d_model, kernel_size=kernel_size, groups=d_model, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)

        def forward(self, x):
            value, gate = self.in_proj(x).chunk(2, dim=-1)
            value = value.transpose(1, 2)                          # [B, D, T] for Conv1d
            value = F.pad(value, (self.kernel_size - 1, 0))        # LEFT padding only -> causal
            value = self.conv(value).transpose(1, 2)               # back to [B, T, D]
            return self.out_proj(value * torch.sigmoid(gate))


    class HybridBlock(nn.Module):
        def __init__(self, d_model, n_heads, ffn_dim, max_seq_len, conv_kernel, use_attention):
            super().__init__()
            self.use_attention = use_attention
            self.norm1 = RMSNorm(d_model)
            self.mixer = (CausalSelfAttention(d_model=d_model, n_heads=n_heads, max_seq_len=max_seq_len)
                          if use_attention else GatedCausalConv(d_model=d_model, kernel_size=conv_kernel))
            self.norm2 = RMSNorm(d_model)
            self.ffn = SwiGLU(d_model=d_model, hidden_dim=ffn_dim)

        def forward(self, x):
            x = x + self.mixer(self.norm1(x))
            x = x + self.ffn(self.norm2(x))
            return x


    class HybridLM(nn.Module):
        def __init__(self, vocab_size, d_model, n_layers, n_heads, ffn_dim, max_seq_len, conv_kernel):
            super().__init__()
            self.vocab_size = vocab_size
            self.d_model = d_model
            self.token_embedding = nn.Embedding(vocab_size, d_model)
            # even layers -> attention, odd layers -> gated conv
            self.blocks = nn.ModuleList([
                HybridBlock(d_model=d_model, n_heads=n_heads, ffn_dim=ffn_dim, max_seq_len=max_seq_len,
                            conv_kernel=conv_kernel, use_attention=(i % 2 == 0))
                for i in range(n_layers)])
            self.final_norm = RMSNorm(d_model)
            self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
            self.lm_head.weight = self.token_embedding.weight      # weight tying
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, input_ids, targets=None):
            x = self.token_embedding(input_ids)
            for block in self.blocks:
                x = block(x)
            x = self.final_norm(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                # FIX: the pipeline's targets are ALREADY shifted by one (y = x[1:]).
                # Your version shifted again (logits[:, :-1] vs targets[:, 1:]) -> the model
                # would learn to predict 2 tokens ahead. Same loss as Models 1-3 now.
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss
    return HybridLM(**cfg)


BUILDERS = {"gpt": build_model1, "llama": build_model2, "gqa": build_model3, "hybrid": build_model4}

# %% 5. ADAPTATION METHODS (from scratch)
class LoRALinear(nn.Module):
    """LoRA:   y = W x + s * B A x,  s = alpha/r.   rsLoRA: same but s = alpha/sqrt(r) (rank-stabilised)."""
    def __init__(self, base, r, alpha, rank_stabilized=False):
        super().__init__()
        self.base = base
        self.scale = alpha / math.sqrt(r) if rank_stabilized else alpha / r
        self.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)


class LoRMALinear(nn.Module):
    """LoRMA (multiplicative):  y = (I + s * B A) W x.  Check the exact variant against the LoRMA paper."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.randn(r, base.out_features) / math.sqrt(base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        y = self.base(x)
        return y + self.scale * F.linear(F.linear(y, self.A), self.B)


class AdaLoRALinear(nn.Module):
    """AdaLoRA:  y = W x + s * P diag(lambda * mask) Q x  (SVD-form update).
    Ranks are pruned during training by importance |lambda * dL/dlambda| (EMA), down to a global budget."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.P = nn.Parameter(torch.randn(base.out_features, r) * 0.02)
        self.Q = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        self.lam = nn.Parameter(torch.zeros(r))
        self.register_buffer("mask", torch.ones(r))
        self.register_buffer("importance", torch.zeros(r))

    def forward(self, x):
        h = F.linear(x, self.Q) * (self.lam * self.mask)
        return self.base(x) + self.scale * F.linear(h, self.P)

    def orth_reg(self):
        r = self.lam.numel()
        I = torch.eye(r, device=self.P.device)
        return ((self.P.T @ self.P - I) ** 2).sum() + ((self.Q @ self.Q.T - I) ** 2).sum()


def wrap_linears(module, fn):
    """replace every nn.Linear `child` (attribute name `name`) by fn(child, name) unless it returns None"""
    for name, child in module.named_children():
        if isinstance(child, nn.Linear):
            new = fn(child, name)
            if new is not None:
                setattr(module, name, new)
        else:
            wrap_linears(child, fn)

class Quant4Linear(nn.Module):
    """Frozen weight stored in 4 bits (QLoRA-style): blockwise absmax, 2 codes packed per byte.
    (QLoRA uses the NF4 code book; this uses uniform 4-bit levels -7..7 - say so in the report.)"""
    def __init__(self, base, block=64):
        super().__init__()
        W = base.weight.data.float()
        self.out_features, self.in_features = W.shape
        flat = W.flatten()
        pad = (-flat.numel()) % block
        flat = torch.cat([flat, flat.new_zeros(pad)]).view(-1, block)
        scale = flat.abs().amax(1, keepdim=True).clamp(min=1e-8) / 7.0
        q = torch.clamp(torch.round(flat / scale), -7, 7).to(torch.int16) + 8      # 1..15
        q = q.flatten().to(torch.uint8)
        if q.numel() % 2: q = torch.cat([q, q.new_zeros(1)])
        self.register_buffer("packed", (q[0::2] | (q[1::2] << 4)).contiguous())
        self.register_buffer("scale", scale.half())
        self.numel, self.block = W.numel(), block
        self.bias = None if base.bias is None else nn.Parameter(base.bias.data.clone(), requires_grad=False)

    def weight(self):
        q = torch.stack([self.packed & 15, self.packed >> 4], 1).flatten()[: self.scale.numel() * self.block]
        W = (q.view(-1, self.block).float() - 8.0) * self.scale.float()
        return W.flatten()[: self.numel].view(self.out_features, self.in_features)

    def forward(self, x):
        return F.linear(x, self.weight().to(x.dtype), None if self.bias is None else self.bias.to(x.dtype))

class IA3Linear(nn.Module):
    """IA3: learned rescaling vector (init 1). mode="out": scales the layer output (keys/values);
    mode="in": scales the input of the FFN down-projection (= the FFN hidden activations)."""
    def __init__(self, base, mode):
        super().__init__()
        self.base, self.mode = base, mode
        self.l = nn.Parameter(torch.ones(base.out_features if mode == "out" else base.in_features))

    def forward(self, x):
        return self.base(x) * self.l if self.mode == "out" else self.base(x * self.l)

IA3_OUT = {"c_attn", "qkv", "k_proj", "v_proj"}   # attention K/V projections (fused qkv: whole output)
IA3_IN = {"proj", "down_proj", "w2"}               # FFN down-projections (their input = FFN activations)


class SubAdapter(nn.Module):
    """Bottleneck adapter on the output of one sub-layer (attention or FFN), before the residual add:
    h = sublayer(x);  h + up(GELU(down(h)))   (up initialised to 0 -> starts as the identity)"""
    def __init__(self, sublayer, d_model, dim):
        super().__init__()
        self.sublayer = sublayer
        self.down, self.up = nn.Linear(d_model, dim), nn.Linear(dim, d_model)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        h = self.sublayer(x)
        return h + self.up(F.gelu(self.down(h)))

ATTN_NAMES = ("attn", "attention", "mixer")    # GPT | LLaMA/GQA | hybrid (attention or gated conv)
FFN_NAMES = ("mlp", "ffn")

def add_adapters(blocks, d_model, dim, where):
    """where="houlsby": adapter after the attention AND the FFN sub-layer of every block (Houlsby 2019)
       where="pfeiffer": adapter only after the FFN sub-layer (Pfeiffer 2020, fewer parameters)"""
    for blk in blocks:
        names = (ATTN_NAMES + FFN_NAMES) if where == "houlsby" else FFN_NAMES
        for n in names:
            if hasattr(blk, n):
                setattr(blk, n, SubAdapter(getattr(blk, n), d_model, dim))

# %% 6. SUMMARISER = Phase-1 LM (+ soft prompt / prefix) with its own tied LM head
def emb_module(bb):
    return bb.wte if hasattr(bb, "wte") else bb.token_embedding

def final_norm(bb):
    return bb.ln_f if hasattr(bb, "ln_f") else bb.final_norm


class SummModel(nn.Module):
    def __init__(self, backbone, d_model, method, n_prompt):
        super().__init__()
        self.bb, self.method = backbone, method
        self.n_prompt = n_prompt if method in ("prompt", "prefix") else 0
        if method == "prompt":
            self.prompt = nn.Parameter(torch.randn(n_prompt, d_model) * 0.02)
        if method == "prefix":
            self.prefix = nn.Parameter(torch.randn(len(backbone.blocks) + 1, n_prompt, d_model) * 0.02)

    def hidden(self, ids):
        bb, P = self.bb, self.n_prompt
        x = emb_module(bb)(ids)
        if P:
            pre = self.prompt if self.method == "prompt" else self.prefix[0]
            x = torch.cat([pre.unsqueeze(0).expand(ids.size(0), -1, -1).to(x.dtype), x], 1)
        if hasattr(bb, "wpe"):
            x = x + bb.wpe(torch.arange(x.size(1), device=ids.device))
        for i, blk in enumerate(bb.blocks):
            if self.method == "prefix" and i > 0:
                x = torch.cat([self.prefix[i].unsqueeze(0).expand(ids.size(0), -1, -1).to(x.dtype), x[:, P:]], 1)
            x = blk(x)
        return final_norm(bb)(x)[:, P:]

    def logits(self, h):
        return self.bb.lm_head(h)


def build_summ_model(model_type, p1_cfg, p1_state, method, keep_every=None):
    bb = BUILDERS[model_type](p1_cfg)
    if p1_state is not None:
        bb.load_state_dict(p1_state)
    if keep_every:
        bb.blocks = nn.ModuleList(list(bb.blocks)[::keep_every])
    d = emb_module(bb).weight.shape[1]
    m = SummModel(bb, d, method, N_PROMPT)
    if method == "full":
        return m
    for p in bb.parameters(): p.requires_grad = False
    if method == "partial":
        for blk in bb.blocks[FREEZE_BOTTOM:]:
            for p in blk.parameters(): p.requires_grad = True
        for p in final_norm(bb).parameters(): p.requires_grad = True
    elif method == "lora":
        wrap_linears(bb.blocks, lambda lin, n: LoRALinear(lin, LORA_R, LORA_ALPHA))
    elif method == "qlora":
        wrap_linears(bb.blocks, lambda lin, n: LoRALinear(Quant4Linear(lin, QUANT_BLOCK), LORA_R, LORA_ALPHA))
    elif method == "lorma":
        wrap_linears(bb.blocks, lambda lin, n: LoRMALinear(lin, LORA_R, LORA_ALPHA))
    elif method == "adalora":
        wrap_linears(bb.blocks, lambda lin, n: AdaLoRALinear(lin, ADA_R_INIT, LORA_ALPHA))
    elif method in ("houlsby", "pfeiffer"):
        add_adapters(bb.blocks, d, ADAPTER_DIM, method)
        if method == "houlsby":                       # Houlsby also trains the layer norms
            for n, p in bb.blocks.named_parameters():
                if "norm" in n or "ln_" in n:
                    p.requires_grad = True
            for p in final_norm(bb).parameters(): p.requires_grad = True
    elif method not in ("prefix", "prompt"):
        raise ValueError(method)
    return m

# %% 7. BUILD MODEL (+ KD teacher)
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16
autocast = lambda: torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP)

RUN_CFG = dict(model_type=MODEL_TYPE, ft_method=FT_METHOD, section=SECTION, tgt_len=TGT_LEN, src_len=SRC_LEN,
               sep_text=SEP_TEXT, lora_r=LORA_R, lora_alpha=LORA_ALPHA, ada_r_init=ADA_R_INIT,
               adapter_dim=ADAPTER_DIM, n_prompt=N_PROMPT, freeze_bottom=FREEZE_BOTTOM,
               keep_every=KD_KEEP_EVERY if KD_TEACHER else None, phase1_cfg=P1_CFG)

def load_run(path):
    """rebuild a saved Phase-6 run (teacher, or RESUME_CKPT) from the run_cfg stored inside it"""
    t = torch.load(as_ckpt_file(path), map_location="cpu", weights_only=False)
    tc = t["run_cfg"]
    for k in ("lora_r", "lora_alpha", "ada_r_init", "adapter_dim", "n_prompt", "freeze_bottom"):
        globals()[k.upper()] = tc[k]
    m = build_summ_model(tc["model_type"], tc["phase1_cfg"], None, tc["ft_method"], tc.get("keep_every"))
    m.load_state_dict(t["state"])
    return m, tc

teacher = None
if KD_TEACHER:
    teacher, tc = load_run(KD_TEACHER)
    assert (tc["section"], tc["src_len"], tc["tgt_len"], tc["sep_text"]) == (SECTION, SRC_LEN, TGT_LEN, SEP_TEXT), \
        f"teacher must read the same input: set SECTION={tc['section']!r}, TGT_LEN={tc['tgt_len']}, N_PROMPT={tc['n_prompt']}"
    teacher = teacher.to(DEVICE).eval()
    for p in teacher.parameters(): p.requires_grad = False
    n_teacher = sum(p.numel() for p in teacher.parameters()) + \
        sum(q.numel for q in teacher.modules() if isinstance(q, Quant4Linear))
    print(f"KD teacher: {tc['model_type']} + {tc['ft_method']} ({n_teacher:,} params)")
    FT_METHOD = RUN_CFG["ft_method"] = "full"

if RESUME_CKPT:
    model, rc = load_run(RESUME_CKPT)
    RUN_CFG = rc
    assert (rc["section"], rc["src_len"]) == (SECTION, SRC_LEN), \
        f"set SECTION={rc['section']!r}, TGT_LEN={rc['tgt_len']}, N_PROMPT={rc['n_prompt']} like the saved run"
    model = model.to(DEVICE)
    EPOCHS = 0
    print(f"RESUME: {rc['model_type']} + {rc['ft_method']} loaded from {RESUME_CKPT} - decoding only")
else:
    model = build_summ_model(MODEL_TYPE, P1_CFG, ck["model"], FT_METHOD, KD_KEEP_EVERY if KD_TEACHER else None).to(DEVICE)
del ck
n_total = sum(p.numel() for p in model.parameters()) + sum(q.numel for q in model.modules() if isinstance(q, Quant4Linear))
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{RUN_CFG['ft_method']}{' (KD student)' if KD_TEACHER else ''}: trainable {n_train:,} / {n_total:,} "
      f"({100 * n_train / n_total:.2f}%) | section={SECTION} | source {SRC_LEN} tok | summary {TGT_LEN} tok")
if KD_TEACHER:
    print(f"KD: student {n_total:,} params = {100 * n_total / n_teacher:.0f}% of teacher")

# %% 8. ROUGE (from scratch) + GENERATION
def words(s):
    return re.findall(r"[a-z0-9]+", s.lower())

def ngram_f1(a, b, n):
    ca = Counter(tuple(a[i:i + n]) for i in range(len(a) - n + 1))
    cb = Counter(tuple(b[i:i + n]) for i in range(len(b) - n + 1))
    ov = sum((ca & cb).values())
    if not ov: return 0.0
    p, r = ov / max(sum(ca.values()), 1), ov / max(sum(cb.values()), 1)
    return 2 * p * r / (p + r)

def lcs_f1(a, b):
    if not a or not b: return 0.0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    l = prev[-1]
    if not l: return 0.0
    p, r = l / len(a), l / len(b)
    return 2 * p * r / (p + r)

def rouge(preds, refs):
    P, R = [words(p) for p in preds], [words(r) for r in refs]
    return {"rouge1": float(np.mean([ngram_f1(p, r, 1) for p, r in zip(P, R)])),
            "rouge2": float(np.mean([ngram_f1(p, r, 2) for p, r in zip(P, R)])),
            "rougeL": float(np.mean([lcs_f1(p, r) for p, r in zip(P, R)]))}

def redundancy(s):
    """share of repeated word trigrams inside one summary (0 = no repetition)"""
    w = words(s); tri = [tuple(w[i:i + 3]) for i in range(len(w) - 2)]
    return 1 - len(set(tri)) / len(tri) if tri else 0.0

def clean(s):
    return re.sub(r"\s+", " ", s.replace("<|endoftext|>", " ")).strip()

@torch.no_grad()
def generate(m, docs, max_new=None, min_new=None):
    """batched decoding without padding: documents sorted by prompt length, every prompt in a batch is cut
    (from the left) to the shortest one - with sorting this almost never removes anything."""
    max_new, min_new = max_new or MAX_NEW, min_new or MIN_NEW
    m.eval()
    out = {}
    order = sorted(docs, key=lambda d: len(d["src"]))
    limit = CTX - m.n_prompt
    for b in range(0, len(order), GEN_BATCH):
        part = order[b:b + GEN_BATCH]
        prompts = [d["src"] + SEP for d in part]
        L = min(map(len, prompts))
        x = torch.tensor([p[-L:] for p in prompts], device=DEVICE)
        gen = [[] for _ in part]
        done = torch.zeros(len(part), dtype=torch.bool, device=DEVICE)
        for t in range(max_new):
            with autocast():
                h = m.hidden(x[:, -limit:])[:, -1]
                logits = m.logits(h).float()
            if REPETITION_PENALTY != 1.0:
                for i, g in enumerate(gen):
                    if g:
                        u = torch.tensor(list(set(g)), device=DEVICE)
                        v = logits[i, u]
                        logits[i, u] = torch.where(v > 0, v / REPETITION_PENALTY, v * REPETITION_PENALTY)
            if NO_REPEAT_NGRAM and t >= NO_REPEAT_NGRAM - 1:
                n = NO_REPEAT_NGRAM
                for i, g in enumerate(gen):
                    pre = tuple(g[-(n - 1):])
                    banned = [g[j + n - 1] for j in range(len(g) - n + 1) if tuple(g[j:j + n - 1]) == pre]
                    if banned:
                        logits[i, banned] = -float("inf")
            if t < min_new:
                logits[:, EOS] = -float("inf")
            if DECODING == "sample":
                probs = torch.softmax(logits / TEMPERATURE, -1)
                sp, si = probs.sort(-1, descending=True)
                sp[(sp.cumsum(-1) - sp) > TOP_P] = 0
                nxt = si.gather(1, torch.multinomial(sp / sp.sum(-1, keepdim=True), 1)).squeeze(1)
            else:
                nxt = logits.argmax(-1)
            nxt = torch.where(done, torch.full_like(nxt, EOS), nxt)
            for i in range(len(part)):
                if not done[i] and nxt[i].item() != EOS:
                    gen[i].append(nxt[i].item())
            done |= nxt == EOS
            x = torch.cat([x, nxt[:, None]], 1)
            if done.all():
                break
        for d, g in zip(part, gen):
            s = clean(tokenizer.decode(g))
            if not s:                                    # never submit an empty summary
                s = " ".join(d["sents"][:3]) if d.get("sents") else "The appeal is disposed of."
            out[d["id"]] = s
        print(f"  generated {min(b + GEN_BATCH, len(order))}/{len(order)}", end="\r")
    print()
    return [out[d["id"]] for d in docs]

# %% 9. TRAINING (teacher forcing, loss only on summary tokens)
def make_batch(docs):
    seqs, labs = [], []
    for d in docs:
        tgt = d["tgt"][:TGT_LEN] + [EOS]
        s = d["src"] + SEP + tgt
        seqs.append(s); labs.append([-100] * (len(s) - len(tgt)) + tgt)
    T = max(map(len, seqs))
    x = torch.full((len(seqs), T), EOS, dtype=torch.long)
    y = torch.full((len(seqs), T), -100, dtype=torch.long)
    for i, (s, l) in enumerate(zip(seqs, labs)):
        x[i, :len(s)] = torch.tensor(s); y[i, :len(l)] = torch.tensor(l)
    return x.to(DEVICE), y.to(DEVICE)

def summary_logits(m, x, y):
    """logits only at positions that predict a summary token (saves memory: vocab is 50k)"""
    with autocast():
        h = m.hidden(x)[:, :-1]
        mask = y[:, 1:] != -100
        return m.logits(h[mask]).float(), y[:, 1:][mask]

ada_mods = [m_ for m_ in model.modules() if isinstance(m_, AdaLoRALinear)]

@torch.no_grad()
def dev_loss(docs):
    model.eval()
    tot, n = 0.0, 0
    for b in range(0, len(docs), MICRO_BATCH):
        x, y = make_batch(docs[b:b + MICRO_BATCH])
        lg, tg = summary_logits(model, x, y)
        tot += F.cross_entropy(lg, tg, reduction="sum").item(); n += tg.numel()
    model.train()
    return tot / max(n, 1)

train_docs = splits["train"]
DEV_LOSS_SET = splits["dev"][:DEV_LOSS_DOCS]
BEST_PATH = os.path.join(OUTPUT_DIR, "best_summ.pt")
history, TRAIN_MIN = [], 0.0
if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()

if EPOCHS > 0:
    new_ids = {id(p) for n, p in model.named_parameters()
               if not n.startswith("bb.") or n.endswith((".A", ".B", ".P", ".Q", ".lam", ".l")) or ".down." in n or ".up." in n}
    groups = [{"params": [p for p in model.parameters() if p.requires_grad and id(p) in new_ids], "lr": LR_NEW},
              {"params": [p for p in model.parameters() if p.requires_grad and id(p) not in new_ids], "lr": LR_BODY}]
    optimizer = torch.optim.AdamW([g for g in groups if g["params"]], weight_decay=WEIGHT_DECAY)
    steps_per_epoch = math.ceil(len(train_docs) / (MICRO_BATCH * GRAD_ACCUM))
    total_steps = EPOCHS * steps_per_epoch
    warmup = max(1, int(0.05 * total_steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda s: min(1.0, (s + 1) / warmup) * max(0.0, (total_steps - s) / max(1, total_steps - warmup)))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)

    ADA_BUDGET0 = sum(m_.lam.numel() for m_ in ada_mods)
    ADA_TARGET = ADA_R_TARGET * len(ada_mods)
    ADA_T0, ADA_T1, ADA_EVERY, ADA_BETA = int(0.1 * total_steps), int(0.7 * total_steps), 10, 0.85
    def adalora_update(step):
        for m_ in ada_mods:
            if m_.lam.grad is not None:
                m_.importance.mul_(ADA_BETA).add_((1 - ADA_BETA) * (m_.lam * m_.lam.grad).abs().detach())
        if step < ADA_T0 or step % ADA_EVERY: return
        prog = min(1.0, (step - ADA_T0) / max(1, ADA_T1 - ADA_T0))
        budget = int(ADA_TARGET + (ADA_BUDGET0 - ADA_TARGET) * (1 - prog) ** 3)
        allimp = torch.cat([m_.importance for m_ in ada_mods])
        if budget < allimp.numel():
            thr = torch.topk(allimp, budget).values[-1]
            for m_ in ada_mods:
                m_.mask.copy_((m_.importance >= thr).float())

    best, step, t0 = float("inf"), 0, time.time()
    EVAL_EVERY = max(1, steps_per_epoch // 2)
    def evaluate_and_maybe_save(tag):
        global best
        dl = dev_loss(DEV_LOSS_SET)
        history.append({"at": tag, "dev_loss": dl, "dev_ppl": math.exp(min(dl, 20))})
        print(f"  [{tag}] dev loss {dl:.4f} | ppl {math.exp(min(dl, 20)):.2f} | {time.time() - t0:.0f}s")
        if dl < best:
            best = dl
            torch.save({"state": model.state_dict(), "run_cfg": RUN_CFG}, BEST_PATH)
            print("  >> best model saved")

    evaluate_and_maybe_save("start")
    for epoch in range(1, EPOCHS + 1):
        model.train()
        random.shuffle(train_docs)
        tot, n = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        micro = [train_docs[i:i + MICRO_BATCH] for i in range(0, len(train_docs), MICRO_BATCH)]
        for i, mb in enumerate(micro):
            x, y = make_batch(mb)
            lg, tg = summary_logits(model, x, y)
            loss = F.cross_entropy(lg, tg)
            if teacher is not None:
                with torch.no_grad():
                    t_lg, _ = summary_logits(teacher, x, y)
                kd = F.kl_div(F.log_softmax(lg / KD_T, -1), F.log_softmax(t_lg / KD_T, -1),
                              log_target=True, reduction="batchmean")
                loss = KD_ALPHA * loss + (1 - KD_ALPHA) * KD_T ** 2 * kd
            if ada_mods:
                loss = loss + 0.1 * sum(m_.orth_reg() for m_ in ada_mods) / len(ada_mods)
            scaler.scale(loss / GRAD_ACCUM).backward()
            tot += loss.item(); n += 1
            if (i + 1) % GRAD_ACCUM == 0 or i + 1 == len(micro):
                scaler.unscale_(optimizer)
                if ada_mods: adalora_update(step)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.grad is not None], 1.0)
                scaler.step(optimizer); scaler.update(); sched.step()
                optimizer.zero_grad(set_to_none=True)
                step += 1
                if step % 25 == 0:
                    rate = (time.time() - t0) / step
                    print(f"epoch {epoch} step {step}/{total_steps} | loss {tot / n:.4f} | "
                          f"ETA training {rate * (total_steps - step) / 60:.0f} min")
                if step % EVAL_EVERY == 0:
                    evaluate_and_maybe_save(f"epoch {epoch} step {step}")
    TRAIN_MIN = (time.time() - t0) / 60
    model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE, weights_only=False)["state"])
    print(f"\nBest dev loss {best:.4f} (ppl {math.exp(min(best, 20)):.2f}) | {TRAIN_MIN:.1f} min")
PEAK_GB = torch.cuda.max_memory_allocated() / 1024**3 if DEVICE.type == "cuda" else None

# %% 10. DEV: generate, ROUGE, length, redundancy (BERTScore is computed by Codabench)
gen_dev = splits["dev"][:GEN_DEV_DOCS]
t1 = time.time()
dev_preds = generate(model, gen_dev)
DEV_MS = (time.time() - t1) / max(len(gen_dev), 1) * 1000
refs = [d["summary"] for d in gen_dev]
DEV_ROUGE = rouge(dev_preds, refs)
pl, rl = [len(p.split()) for p in dev_preds], [len(r.split()) for r in refs]
DEV_STATS = {**DEV_ROUGE, "pred_words_mean": float(np.mean(pl)), "ref_words_mean": float(np.mean(rl)),
             "redundancy": float(np.mean([redundancy(p) for p in dev_preds])),
             "ref_redundancy": float(np.mean([redundancy(r) for r in refs])), "ms_per_doc": DEV_MS}
print(f"DEV ({len(gen_dev)} docs): ROUGE-1 {DEV_ROUGE['rouge1']:.4f} | ROUGE-2 {DEV_ROUGE['rouge2']:.4f} | "
      f"ROUGE-L {DEV_ROUGE['rougeL']:.4f}")
print(f"  length: generated {np.mean(pl):.0f} words vs reference {np.mean(rl):.0f} | repeated trigrams "
      f"{100 * DEV_STATS['redundancy']:.1f}% (reference {100 * DEV_STATS['ref_redundancy']:.1f}%) | {DEV_MS:.0f} ms/doc")
for p, r in list(zip(dev_preds, refs))[:2]:
    print("\nREFERENCE:", r[:500], "\nGENERATED:", p[:500])

# %% 11. TEST SUMMARIES + prediction.json + ZIP
t1 = time.time()
test_preds = generate(model, splits["test"])
INFER_MS = (time.time() - t1) / len(splits["test"]) * 1000
preds = [{"id": d["id"], "summary": s} for d, s in zip(splits["test"], test_preds)]
assert len(preds) == len(splits["test"]) == len({p["id"] for p in preds})
assert all(isinstance(p["summary"], str) and p["summary"] for p in preds)
print(f"{len(preds)} test summaries | mean {np.mean([len(p['summary'].split()) for p in preds]):.0f} words | "
      f"{INFER_MS:.0f} ms per doc\nexample: {preds[0]['summary'][:300]}")

pred_path = os.path.join(OUTPUT_DIR, "prediction.json")
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(preds, f, ensure_ascii=False)
zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname="prediction.json")
print("Submission ready:", zip_path)

# %% 12. RUN SUMMARY (experiment log / report)
with open(os.path.join(OUTPUT_DIR, "run_summary.json"), "w") as f:
    json.dump({**{k: v for k, v in RUN_CFG.items() if k != "phase1_cfg"}, "kd_teacher": KD_TEACHER,
               "resume_ckpt": RESUME_CKPT, "trainable_params": n_train, "total_params": n_total,
               "teacher_params": n_teacher if KD_TEACHER else None, "train_docs": len(train_docs), "epochs": EPOCHS,
               "decoding": dict(mode=DECODING, max_new=MAX_NEW, min_new=MIN_NEW, rep_penalty=REPETITION_PENALTY,
                                no_repeat_ngram=NO_REPEAT_NGRAM, top_p=TOP_P, temperature=TEMPERATURE),
               "history": history, "dev": DEV_STATS, "train_minutes": round(TRAIN_MIN, 1), "peak_gpu_gb": PEAK_GB,
               "test_ms_per_doc": round(INFER_MS, 1)}, f, indent=2)
print("Done.")

Device: cuda Tesla T4


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

/tmp/cs781_data/Phase_6_Summ/Test/test.json 2.7 MB
/tmp/cs781_data/Phase_6_Summ/Train/train.json 210.6 MB

Test/test.json -> test: list with 100 entries | first: {"id": "4963", "document": "Tax Reference Cases Nos. 3 to 5 of 1975 Tax Reference under Section 257 of the Income tax Act, 1961 made by the Income tax Appellate Tribunal, Cuttack Bench, Cuttack P.A. Francis, Champat Rai and Miss A. Subhashini for the Appellant. C.S.S. Rao for the Respondent. The Jud...

Train/train.json -> train: list with 7030 entries | first: {"id": "6712", "document": "IVIL Appeal No.768 (NT) of 1977 etc. From the judgment Order dated 9.10.1975 of the Madhya Pradesh High Court in M.C.C. No.144 of 1966. V.A. Bobde, B.R.Aggarwala and U.A.Rana for the Appellant. S.V.Deshpande and S.K.Agnihotri for the Respondents. The Judgment of the Court...

no dev file - held out 300 training documents as dev
train: 6728 docs | document words median 3129, max 139645 | summary words median 638, p90 1494
dev: 300 docs | docum